In [ ]:
import matplotlib.pyplot as plt

def image_show(imagem):
    """
    Exibe uma imagem usando matplotlib.
    :param imagem: Objeto de imagem PIL
    """
    plt.imshow(imagem)
    plt.show()


In [ ]:
import torchvision.models as models 
import torch

def resnet_101(num_class):
    # Create instace of resnet101
    model = models.resnet101(weights = "ResNet101_Weights.DEFAULT")
    # Take total features from the last layer
    num_features = model.fc.in_features
    # Create my own layer with num_class do i want
    model.fc = torch.nn.Linear(num_features, num_class)
    # Multi category cross entropy(Funcao de erro)
    # Binario e multi categorico
    return model    

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from PIL import Image

class CustomDataset(Dataset):
    def verify_image(self, image_path):
        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception as e:
            print(f"[ERRO] Imagem inválida: {image_path} - {e}")


    def __init__(self, root_dir, transforms=None):
        # Define the inital variables
        self.root_dir = root_dir
        self.transforms = transforms
        # List sub floder for define class 
        self.classes = os.listdir(root_dir)
        self.classes.sort()
        print("classes name:", self.classes)
        self.images = []
        # For in each sub folder and get the images name folders
        for clc in self.classes:
            images_name = os.listdir(self.root_dir + "/" + clc)
            self.images += [self.root_dir + "/" + clc + "/" + img_name for img_name in images_name]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        # Take path with index
        image_path = self.images[index]
        # Read image and convert to RGB
        self.verify_image(image_path)
        image = Image.open(image_path).convert("RGB")
        if self.transforms:
            image = self.transforms(image)
        image_class_name = image_path.split("/")[-2]
        label = torch.tensor(self.classes.index(image_class_name))
        return image, label


class TrainDatasetImplemetation:
    
    def __init__(self, data_path, image_size, batch_size=16):
        # Convert image to PIL
        self.to_pill = transforms.ToPILImage()
        # Resize image to the size parameter
        self.resize_img = transforms.Resize(image_size)
        # Create color variation for import ia 
        self.color_jitter = transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)
        # Horizontally flips the image with a 50% probability 
        self.flip_horizontal = transforms.RandomHorizontalFlip(p=0.5)
        # Vertically flips the image with a 10% probability.
        self.flip_vertical = transforms.RandomVerticalFlip(p=0.1)
        # Rotates the image randomly within ±30 degrees.
        self.random_rotation = transforms.RandomRotation(degrees=30)
        # Converts the image to a tensor for PyTorch models.
        self.to_tensor = transforms.ToTensor()
        # Create the train and test dataset path
        self.train_dataset_path = data_path + "/train"
        self.test_dataset_path = data_path + "/test"
        # Create the batch size
        self.batch_size = batch_size

    def report_size_img(self, train_data, test_data):
        for data in train_data:
            images, labels = data
            print("train images shape:", images.shape)
            print("train labels:", labels)
            break

        for data in test_data:
            images, labels = data
            print("test images shape:", images.shape)
            print("test labels:", labels)
            break

    def load_data(self):

        transforms_train = transforms.Compose([ 
                                            self.resize_img, self.color_jitter, 
                                            self.flip_horizontal, self.flip_vertical, 
                                            self.random_rotation, self.to_tensor
                                            ])
        
        transforms_test = transforms.Compose([
                                            self.resize_img,
                                            self.to_tensor
                                            ])
        # Implamete custom dataset loader
        train_dataset = CustomDataset(self.train_dataset_path, transforms= transforms_train)
        test_dataset = CustomDataset(self.test_dataset_path, transforms=transforms_test)

        print("no of samples in train dataset", len(train_dataset))
        print("no of samples in test dataset", len(test_dataset))
        
        '''
        DataLoader makes it easy to efficiently load data in batches, 
        allows shuffling the data to improve training, 
        and supports parallel loading to speed up data preparation during model training.
        '''
        train_loader = DataLoader(train_dataset, batch_size= self.batch_size, shuffle= True)
        test_loader = DataLoader(test_dataset, batch_size= self.batch_size, shuffle= True)
        self.report_size_img(train_loader, test_loader)
        return train_loader, test_loader
    
    def save_image(self):
        train_data, test_data = self.load_data()
        self.report_size_img(train_data, test_data)

        for data in train_data:
            images, labels = data
            save_image(images, "images_train_101.jpg")
            break

        for data in test_data:
            images, labels = data
            print("test images shape:", images.shape)
            print("test labels:", labels)
            save_image(images, "images_test_101.jpg")
            break



In [ ]:
import os 
from time import sleep
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
from tqdm import tqdm
# TrainDatasetImplemetation
# from models.resnet import resnet_50 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   

print("Dispositivo utilizado: ", device)

In [ ]:
data_set_path = "../imagens"    

image_size = (224, 224)
batch_size = 32
data_set = TrainDatasetImplemetation(data_set_path, image_size, batch_size)
train_data, test_data = data_set.load_data()

In [ ]:
# Define binary classficiation!
num_classes = 2
# Create the model implementation
model = resnet_101(num_classes)
# Pass model to the device
model = model.to(device)
# Train model 
model.train()

In [ ]:
# Define the learnin rate
learning_rate = 1e-3
# Number of epochs
num_epochs = 30
'''
    Define the loss function for multi-class classification.
    CrossEntropyLoss combines LogSoftmax and Negative Log-Likelihood Loss (NLLLoss) in one single class.
    It expects raw, unnormalized logits as input and automatically applies softmax.
    The target should contain class indices (e.g., 0, 1, 2...) corresponding to the correct class.

    Internally, for each prediction:
    1. Applies softmax to convert logits into probabilities.
    2. Takes the log of the probability for the correct class.
    3. Applies the negative log to calculate the loss.
    Intuition:
    - If the model gives high probability to the correct class → low loss (good).
    - If the model gives low probability to the correct class → high loss (bad).
    Used for classification problems with two or more classes.
'''
criterion = nn.CrossEntropyLoss()
'''
    Define the optimizer to update the model's parameters during training.
    Adam (Adaptive Moment Estimation) is an optimization algorithm that combines
    the benefits of AdaGrad and RMSProp. It adapts the learning rate for each parameter
    using estimates of the first and second moments of the gradients.

    Arguments:
    - model.parameters(): passes all trainable parameters of the model to the optimizer.
    - lr=learning_rate: sets the initial learning rate for updating the weights.

    Adam is widely used because it typically converges faster and requires less tuning
    of the learning rate compared to traditional stochastic gradient descent (SGD).
'''
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
def get_model_accuracy(test_data, model):
    '''
        Evaluate the model's performance on the test data and print a classification report.

        Arguments:
        - test_data: a DataLoader containing the test dataset.
        - model: the trained model to evaluate.
    '''
    # Initialize arrays to store predicted and true labels
    y_pred = np.zeros(0)
    y_true = np.zeros(0)
    
    # Set the model to evaluation mode (disables dropout, batch norm updates, etc.)
    model.eval()
    # Disable gradient computation to save memory and improve performance
    with torch.no_grad():
        for images_batch, y_true_batch in test_data:

            images_batch = images_batch.to(device)

            scores = model(images_batch)
            _, y_pred_batch = scores.max(1)
            y_pred_batch = y_pred_batch.cpu()

            y_pred = np.concatenate((y_pred, y_pred_batch))
            y_true = np.concatenate((y_true, y_true_batch))

    model.train()

    report = classification_report(y_true, y_pred)
    print(report)

In [ ]:
for epoch in range(1, num_epochs + 1):
    # Create a tqdm progress bar for visualizing training progress per batch
    pbar_batch = tqdm(train_data, unit="batch")
    losses = []
    # Iterate over each batch of data
    for data in pbar_batch:
        # Set the description of the progress bar to show current epoch

        pbar_batch.set_description(f"Epoch {epoch}")
        # Unpacking and sending to GPU
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        # raw scores before softmax
        scores = model(images)
        # We calculate the loss using criterion (CrossEntropyLoss).
        loss = criterion(scores, labels)
        losses.append(loss.item())
        cost = sum(losses)/len(losses)

        # Backward Pass and Weight Update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Updates the progress bar
        pbar_batch.set_postfix(loss=cost)
        sleep(0.1)
    '''
        Every 5 epochs, the model is evaluated with the 
        test data (test_data) using the get_model_accuracy() function.
    '''
    if (epoch % 5) == 0:
        get_model_accuracy(test_data, model)
        sleep(0.1)


print("model training complete")

In [ ]:
'''
    saves the trained model weights to disk, creating the directory 
    if it does not already exist.
'''
model_save_path = "weights/resnet"
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

torch.save(model.state_dict(), model_save_path + "/resnet_model_checkpoint.pth")